Preprocessing by Nguyen

In [ ]:
import ast
import pandas as pd
import numpy as np
import spacy
from typing import Literal
import pickle

class Preprocess:
    def __init__(self):
        pass

Text = tuple[Literal[-1, 1], np.ndarray]
Dialog = list[Text]

def get_embeddings(file: str) -> dict:
    embeddings = {}
    with open(file, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], "float32")
            embeddings[word] = vector
    return embeddings

def preprocess_data(file: str = "Data/train.csv", embeddings_file: str = "vectors/glove.6B.50d.txt") -> list[Dialog]:
    embeddings = get_embeddings(embeddings_file)
    print("Embeddings loaded with size:", len(embeddings))
    nlp = spacy.load("en_core_web_sm")
    df = pd.read_csv(file)
    data: list[Dialog] = []
    for row in df.iloc:
        dialogue = ast.literal_eval(row['Dialogue'])['text']
        dialogue_data: Dialog = []
        for text in dialogue:
            if text['response'] == '$S$':
                continue
            if text['response'] == '$EXIT$':
                break
            role = 1 if text['role'] == 'A' else -1
            doc = nlp(text['response'])
            embeddings_data: list[np.ndarray] = []
            for token in doc:
                if token.lemma_ in embeddings:
                    embeddings_data.append(embeddings[token.lemma_])
            embeddings_data = np.array(embeddings_data)
            dialogue_data.append((role, embeddings_data))
        data.append(dialogue_data)
    with open("features.pkl", "wb") as f:
        pickle.dump(data, f)
    return data

def load_data(file: str = "features.pkl") -> list[Dialog]:
    with open(file, "rb") as f:
        data = pickle.load(f)
    return data

Preprocessing by Yibai

In [ ]:
from torch.nn.utils.rnn import PackedSequence

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
def to_inputs_and_labels(data: list[Dialog]) -> tuple[list[list[np.ndarray]], list[list[Literal[-1, 1]]]]:
    '''Split data into inputs and labels

    Args:
        data(list[Dialog]): loaded training data, with non-uniform batch size and sequence length.  
                
    :Returns: tuple[inputs, labels] WHERE

        inputs(list[list[np.ndarray]]): input data, with non-uniform batch size and sequence length.
        
        labels(list[list[Literal[-1, 1]]]): labels for each sequence, with non-uniform batch size and sequence length.
    '''
    inputs = []
    labels = []
    for batch in data:
        inputs_batch = []
        labels_batch = []
        for pair in batch:
            inputs_batch.append(pair[1])
            labels_batch.append(pair[0])
        inputs.append(inputs_batch)
        labels.append(labels_batch)
    return (inputs, labels)

In [ ]:
def is_valid_batch(
    inputs: list[np.ndarray],
    labels: list[Literal[-1, 1]],
    hidden_size: int
) -> bool:
    '''Verify whether the inputs and labels of a batch have valid vector representations.

    Args:
        inputs(list[np.ndarray]): list of all sequences in the batch
        labels(list[Literal[-1, 1]]): list of labels for each sequence in the batch
        hidden_size(int): hidden dimension size
    '''
    # Verify that inputs and labels are non-empty:
    if len(inputs) == 0 or len(labels) == 0:
        return False
    
    # Verify that all sequences in the batch are non-empty:
    for seq in inputs:
        if seq.shape == (0, ):
            return False
    
    # Verify that all tokens in the batch have the expected size of embedding:
    for seq in inputs:
        if seq.shape[1] != hidden_size:
            return False
    
    # Verify that inputs and labels have compatible dimensions:
    if len(inputs) != len(labels):
        return False

    return True

def filter_good_batches(
    inputs: list[list[np.ndarray]],
    labels: list[list[Literal[-1, 1]]],
    hidden_size: int
) -> tuple[list[list[np.ndarray]], list[list[Literal[-1, 1]]]]:
    '''Remove all batches containing invalid data

    Args:
        inputs(list[list[np.ndarray]]): inputs for all batches
        labels(list[list[Literal[-1, 1]]]): labels every sequences
    '''
    inputs_copy = inputs.copy()
    labels_copy = labels.copy()
    num_batch = len(inputs)
    assert(num_batch == len(labels))
    
    pop_count = 0
    for i in range(num_batch):
        if not is_valid_batch(inputs[i], labels[i], hidden_size):
            inputs_copy.pop(i - pop_count)
            labels_copy.pop(i - pop_count)
            pop_count += 1
    
    return (inputs_copy, labels_copy)

In [ ]:
def pad_inputs(inputs: list[list[np.ndarray]]) -> list[PackedSequence]:
    '''Pad the sequences with tokens, followed by padding batches with sequences, to align the dimensions.

    Returns:
        updated_inputs(list(PackedSequence)): Input in processable format by pytorch rnn modules.
    '''
    updated_inputs = []
    for batch in inputs:
        # Prepare mask for packing padded sequence
        seq_lens = [len(seq) for seq in batch]

        # Batch format: from list[np.ndarray] to list[torch.Tensor]
        batch = [torch.Tensor(seq) for seq in batch]
        
        padded_batch = nn.utils.rnn.pad_sequence(batch, padding_side="left")
        packed_batch = nn.utils.rnn.pack_padded_sequence(padded_batch, seq_lens, enforce_sorted=False)
        updated_inputs.append(packed_batch)
    return updated_inputs

In [ ]:
data = load_data("../features.pkl")
HIDDEN_SIZE = 50

inputs, labels = to_inputs_and_labels(data)
inputs, labels = filter_good_batches(inputs, labels, HIDDEN_SIZE)
inputs = pad_inputs(inputs)

Simple LSTM Model

In [ ]:
INPUT_SIZE = 50
HIDDEN_SIZE = 32

In [ ]:
class LstmBasic(nn.Module):
    def __init__(self, input_size, hidden_size) -> None:
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size)

    def forward(self, x):
        x = self.lstm(x)
        return x

In [ ]:
def train(model, inputs, labels):
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

    output = model(inputs[0])
    loss = loss_function(output[0], labels[0])
    loss.backward()
    optimizer.step()

    return loss.item()

In [ ]:
model = LstmBasic(INPUT_SIZE, HIDDEN_SIZE)
train(model, inputs, labels)